In [1]:
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from typing import Tuple

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.vis as vis

from config_handlers.eval_handler import EvalHandler
from config_handlers.rk_handler import RKHandler

In [2]:
rk_hdl = RKHandler()
eval_hdl = EvalHandler()

model_list = rk_hdl.get_model_list()
rk_hdl.set_curr_model(model_list[0])
ds_list = rk_hdl.get_ds_list()

all_ds = ds_list * 4
all_models = model_list * 4
all_models.sort()

MC_DATASETS = ["seed", "vqav2"]
OPEN_DATASETS = ["llava-bench", "mmbench"]

CHARTS_DIR = Path("behaviour_charts")
data_io.make_dir(CHARTS_DIR)

[Errno 17] File exists: 'behaviour_charts'


In [3]:
def load_data(source: str = "greedy") -> Tuple[dict, dict]:
    samples = {m: {ds: {} for ds in ds_list} for m in model_list}
    summaries = {m: {ds: {} for ds in ds_list} for m in model_list}

    for model, ds in zip(all_models, all_ds):
        if source == "nucleus":
            base_fp = Path("0_stats", "similarity_nucleus")
        else:
            base_fp = Path("0_stats", "similarity")

        samples[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "samples.json")
        )
        summaries[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "summary.json")
        )

    return samples, summaries


def load_eval_responses():
    data = {m: {ds: {}} for m in model_list for ds in ds_list}
    for model, ds in zip(all_models, all_ds):
        eval_hdl.set_curr_model(model)
        eval_hdl.set_curr_ds(ds)
        data[model][ds] = data_io.load_json(Path("..", eval_hdl.get_eval_path()))
    return data

In [4]:
eval_data = load_eval_responses()

Check Accuracy (MC QA) and BERTScore F1 (open-ended QA)

In [5]:
eval_data_small = {m: {ds: 0.0 for ds in eval_data[m]} for m in eval_data}
for m in eval_data:
    print(f">> Eval for {m.upper()}")
    for ds in eval_data[m]:
        if "f1" in eval_data[m][ds]["measures"]:
            print(f" - BERTScore F1 on {ds}: {eval_data[m][ds]['measures']['f1']:.3f}")
        elif "accuracy" in eval_data[m][ds]["measures"]:
            print(
                f" - MC accuracy on {ds}: {eval_data[m][ds]['measures']['accuracy']:.3f}"
            )

>> Eval for INTERNVL2
 - MC accuracy on vqav2: 0.660
 - BERTScore F1 on llava-bench: 0.673
 - BERTScore F1 on mmbench: 0.796
 - MC accuracy on seed: 0.787
>> Eval for LLAVA-1.6
 - MC accuracy on vqav2: 0.587
 - BERTScore F1 on llava-bench: 0.669
 - BERTScore F1 on mmbench: 0.793
 - MC accuracy on seed: 0.687
>> Eval for MINIGPT4
 - MC accuracy on vqav2: 0.260
 - BERTScore F1 on llava-bench: 0.634
 - BERTScore F1 on mmbench: 0.632
 - MC accuracy on seed: 0.127
>> Eval for SHAREGPT4V
 - MC accuracy on vqav2: 0.640
 - BERTScore F1 on llava-bench: 0.690
 - BERTScore F1 on mmbench: 0.706
 - MC accuracy on seed: 0.660


In [6]:
from matplotlib import pyplot as plt


def plot_bar(data, model, dataset, out_path):
    width = 0.5
    fig, ax = plt.subplots()
    fig.set_figwidth(20)
    # q_ids = list(data.keys())
    q_ids = list(np.arange(len(data.keys())))
    scores = list(data.values())

    p = ax.bar(q_ids, scores, width)

    # Add lines and counts for each threshold
    for y in np.arange(0.2, 1, 0.2):
        count = len([s for s in scores if s >= y])
        ax.text(-2, y, count, transform=ax.get_xaxis_transform())
        ax.axhline(y=y, color="r", linestyle=":")

    count = len([s for s in scores if s >= 0.7])
    ax.text(-2, 0.7, count, transform=ax.get_xaxis_transform())
    ax.axhline(y=0.7, color="g", linestyle="--")

    ax.set_title(f"BERTScore (F1) for {model.upper()} on {dataset.upper()}")
    # ax.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


# Prepare a bit the data
bert_scores = {model: {} for model in model_list}

for model in eval_data:
    for ds in OPEN_DATASETS:
        bert_scores[model][ds] = []
        for r in eval_data[model][ds]["ranges"]:
            bert_scores[model][ds].extend(eval_data[model][ds]["ranges"][r])

for model in bert_scores:
    for ds in bert_scores[model]:
        bert_scores[model][ds] = {
            question["question_id"]: question["measures"]["f1"]
            for question in bert_scores[model][ds]
        }

for model in bert_scores:
    for ds in bert_scores[model]:
        out_dir = Path("open_ended_perf")
        data_io.make_dir(out_dir)
        out_path = out_dir.joinpath(f"{model}_{ds}.pdf")
        plot_bar(bert_scores[model][ds], model, ds, out_path)

[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'


Plot all similarity values

In [7]:
# det: data obtained with greedy decoding
# sam: data obtained with nucleus sampling decoding
sources = ["greedy", "nucleus"]
measures = ["ged", "iou_nodes", "iou_edges", "cosine"]
samples_greedy, summaries_greedy = load_data(source="greedy")
samples_nucleus, summaries_nuclues = load_data(source="nucleus")

for model, ds in zip(all_models, all_ds):
    print("=" * 10, f"{model.upper()} + {ds.upper()}", "=" * 10)
    print("\t\t = Greedy = \t = Nucleus =")

    # Get GED
    avg_ged_det = summaries_greedy[model][ds]["ged_avg"]
    avg_ged_sam = summaries_nuclues[model][ds]["ged_avg"]
    print(f"Avg. GED: \t {avg_ged_det:.2f} \t\t {avg_ged_sam:.2f}")
    print("-" * 50)

    # Get IOU
    avg_iou_nodes_det = summaries_greedy[model][ds]["iou"]["iou_nodes_avg"]
    avg_iou_edges_det = summaries_greedy[model][ds]["iou"]["iou_edges_avg"]
    avg_iou_nodes_sam = summaries_nuclues[model][ds]["iou"]["iou_nodes_avg"]
    avg_iou_edges_sam = summaries_nuclues[model][ds]["iou"]["iou_edges_avg"]
    print(f"Avg. IOU (n): \t {avg_iou_nodes_det:.2f} \t\t {avg_iou_edges_det:.2f}")
    print(f"Avg. IOU (e): \t {avg_iou_edges_sam:.2f} \t\t {avg_iou_edges_sam:.2f}")
    print("-" * 50)

    # Get Cosine
    avg_cosine_det = summaries_greedy[model][ds]["cosine_avg"]
    avg_cosine_sam = summaries_nuclues[model][ds]["cosine_avg"]
    print(f"Avg. Cosine: \t {avg_cosine_det:.2f} \t\t {avg_cosine_sam:.2f}")
    print("-" * 50)

========== INTERNVL2 + LLAVA-BENCH ==========
		 = Greedy = 	 = Nucleus =
Avg. GED: 	 27.10 		 27.26
--------------------------------------------------
Avg. IOU (n): 	 0.05 		 0.02
Avg. IOU (e): 	 0.02 		 0.02
--------------------------------------------------
Avg. Cosine: 	 0.56 		 0.57
--------------------------------------------------
========== INTERNVL2 + MMBENCH ==========
		 = Greedy = 	 = Nucleus =
Avg. GED: 	 22.28 		 22.63
--------------------------------------------------
Avg. IOU (n): 	 0.09 		 0.02
Avg. IOU (e): 	 0.02 		 0.02
--------------------------------------------------
Avg. Cosine: 	 0.60 		 0.59
--------------------------------------------------
========== INTERNVL2 + SEED ==========
		 = Greedy = 	 = Nucleus =
Avg. GED: 	 22.39 		 22.50
--------------------------------------------------
Avg. IOU (n): 	 0.08 		 0.04
Avg. IOU (e): 	 0.03 		 0.03
--------------------------------------------------
Avg. Cosine: 	 0.54 		 0.55
------------------------------------------

In [9]:
# Visualising measures
sim_charts_dir = Path("similarity_charts")
data_io.make_dir(sim_charts_dir)
for s in sources:
    for measure in measures:
        out_path = sim_charts_dir.joinpath(f"{measure}_{s}.pdf")
        if s == "greedy":
            vis.plot_similarity(samples_greedy, out_path, measure_key=measure)
        elif s == "nucleus":
            vis.plot_similarity(samples_nucleus, out_path, measure_key=measure)
        else:
            print("Problem!")

Proceeding with data from greedy decoding

In [10]:
samples, summaries = load_data(source="greedy")

Check sample-level similarity values

In [11]:
for m, ds in zip(all_models, all_ds):
    print(f">> Results for {m.upper()} on {ds.upper()}")
    print(f"- Num. entries: {len(samples[m][ds])}")
    print(f"- Avg. SIM: {summaries[m][ds]['cosine_avg']:.3f}")
    # print(f"- Std. SIM: {summaries[m][ds]['cosine_std']}")
    print("-" * 30)

>> Results for INTERNVL2 on LLAVA-BENCH
- Num. entries: 58
- Avg. SIM: 0.565
------------------------------
>> Results for INTERNVL2 on MMBENCH
- Num. entries: 139
- Avg. SIM: 0.598
------------------------------
>> Results for INTERNVL2 on SEED
- Num. entries: 150
- Avg. SIM: 0.544
------------------------------
>> Results for INTERNVL2 on VQAV2
- Num. entries: 150
- Avg. SIM: 0.601
------------------------------
>> Results for LLAVA-1.6 on LLAVA-BENCH
- Num. entries: 44
- Avg. SIM: 0.520
------------------------------
>> Results for LLAVA-1.6 on MMBENCH
- Num. entries: 133
- Avg. SIM: 0.573
------------------------------
>> Results for LLAVA-1.6 on SEED
- Num. entries: 129
- Avg. SIM: 0.534
------------------------------
>> Results for LLAVA-1.6 on VQAV2
- Num. entries: 142
- Avg. SIM: 0.548
------------------------------
>> Results for MINIGPT4 on LLAVA-BENCH
- Num. entries: 54
- Avg. SIM: 0.485
------------------------------
>> Results for MINIGPT4 on MMBENCH
- Num. entries: 102
- 

In [ ]:
vis.plot_similarity(
    samples, CHARTS_DIR.joinpath("distrib_similarity.pdf"), measure_key="cosine"
)

Mapping behaviours types

In [9]:
df = pd.DataFrame()
for m, ds in zip(all_models, all_ds):
    for id_q in samples[m][ds]:
        row = {
            "question_id": int(id_q),
            "model": m,
            "dataset": ds,
            "similarity": [samples[m][ds][id_q]["cosine"]],
        }
        temp_df = pd.DataFrame.from_dict(row)
        df = pd.concat([df, temp_df], ignore_index=True)

In [30]:
counts_model = {m: {ds: len(samples[m][ds]) for ds in samples[m]} for m in samples}
counts_model

{'internvl2': {'llava-bench': 58, 'mmbench': 139, 'seed': 150, 'vqav2': 150},
 'llava-1.6': {'llava-bench': 44, 'mmbench': 133, 'seed': 129, 'vqav2': 142},
 'minigpt4': {'llava-bench': 54, 'mmbench': 102, 'seed': 141, 'vqav2': 123},
 'sharegpt4v': {'llava-bench': 49, 'mmbench': 135, 'seed': 109, 'vqav2': 21}}

In [ ]:
for m, ds in zip(all_models, all_ds):
    print(f"{m} on {ds}: {len(df[(df['model'] == m) & (df['dataset'] == ds)])}")

internvl2 on llava-bench: 58
internvl2 on mmbench: 139
internvl2 on seed: 150
internvl2 on vqav2: 150
llava-1.6 on llava-bench: 44
llava-1.6 on mmbench: 133
llava-1.6 on seed: 129
llava-1.6 on vqav2: 142
minigpt4 on llava-bench: 54
minigpt4 on mmbench: 102
minigpt4 on seed: 141
minigpt4 on vqav2: 123
sharegpt4v on llava-bench: 49
sharegpt4v on mmbench: 135
sharegpt4v on seed: 109
sharegpt4v on vqav2: 21


In [10]:
df.head()

,question_id,model,dataset,similarity
0,0,internvl2,llava-bench,0.655310
1,1,internvl2,llava-bench,0.716396
2,2,internvl2,llava-bench,0.588038
3,3,internvl2,llava-bench,0.457952
4,4,internvl2,llava-bench,0.423745


In [15]:
eval_data

{'internvl2': {'vqav2': {'measures': {'accuracy': 0.66},
   'correct_resp': [{'question_id': 337814001,
     'question': 'How many people are shown?',
     'class': 'how_many_people_are',
     'img': '337814',
     'img_path': './data/datasets/vqav2/imgs/how_many_people_are/337814.jpg',
     'is_open': True,
     'options': None,
     'answer': '3',
     'response': '3'},
    {'question_id': 60596018,
     'question': 'How many people are on the bed?',
     'class': 'how_many_people_are',
     'img': '60596',
     'img_path': './data/datasets/vqav2/imgs/how_many_people_are/60596.jpg',
     'is_open': True,
     'options': None,
     'answer': '1',
     'response': '1'},
    {'question_id': 58157004,
     'question': 'How many people are facing away from the players?',
     'class': 'how_many_people_are',
     'img': '58157',
     'img_path': './data/datasets/vqav2/imgs/how_many_people_are/58157.jpg',
     'is_open': True,
     'options': None,
     'answer': '1',
     'response': '1'},

In [11]:
# Preparing the lists of question_ids, organised in buckets
flat_ids = {model: {ds: {} for ds in eval_data[model]} for model in eval_data}

for model, ds in zip(all_models, all_ds):
    if ds in OPEN_DATASETS:
        # Parse ranges
        ranges = list(eval_data[model][ds]["ranges"].keys())
        for r in ranges:
            flat_ids[model][ds][r] = [
                int(q["question_id"]) for q in eval_data[model][ds]["ranges"][r]
            ]
    elif ds in MC_DATASETS:
        # Parse correct_resp and wrong_resp
        flat_ids[model][ds]["correct_resp"] = [
            int(q["question_id"]) for q in eval_data[model][ds]["correct_resp"]
        ]
        flat_ids[model][ds]["wrong_resp"] = [
            int(q["question_id"]) for q in eval_data[model][ds]["correct_resp"]
        ]
    else:
        print("Error.")
    # Parse invalid_resp
    flat_ids[model][ds]["invalid_resp"] = [
        int(q["question_id"]) for q in eval_data[model][ds]["invalid_resp"]
    ]
    print(f"Processed {model} on {ds}.")

Processed internvl2 on llava-bench.
Processed internvl2 on mmbench.
Processed internvl2 on seed.
Processed internvl2 on vqav2.
Processed llava-1.6 on llava-bench.
Processed llava-1.6 on mmbench.
Processed llava-1.6 on seed.
Processed llava-1.6 on vqav2.
Processed minigpt4 on llava-bench.
Processed minigpt4 on mmbench.
Processed minigpt4 on seed.
Processed minigpt4 on vqav2.
Processed sharegpt4v on llava-bench.
Processed sharegpt4v on mmbench.
Processed sharegpt4v on seed.
Processed sharegpt4v on vqav2.


In [12]:
flat_ids["internvl2"]["seed"].keys()

dict_keys(['correct_resp', 'wrong_resp', 'invalid_resp'])

In [13]:
def get_outcome(x, flat_ids):
    for k, v in flat_ids[x["model"]][x["dataset"]].items():
        if int(x["question_id"]) in v:
            # Fix outcome for MC questions
            if x["dataset"] in MC_DATASETS:
                return "correct" if k == "correct_resp" else "wrong"
            else:
                return k
    return None

In [14]:
df["outcome"] = df.apply(lambda x: get_outcome(x, flat_ids), axis=1)

In [16]:
df.head()

,question_id,model,dataset,similarity,outcome
0,0,internvl2,llava-bench,0.655310,60_80
1,1,internvl2,llava-bench,0.716396,60_80
2,2,internvl2,llava-bench,0.588038,60_80
3,3,internvl2,llava-bench,0.457952,60_80
4,4,internvl2,llava-bench,0.423745,40_60


In [ ]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,similarity,outcome
197,3151,internvl2,seed,0.562908,correct
198,1656,internvl2,seed,0.448091,correct
199,154,internvl2,seed,0.711095,correct
200,3057,internvl2,seed,0.765221,correct
201,2173,internvl2,seed,0.429385,correct


In [17]:
df["question_type"] = df.apply(
    lambda x: "open" if x["dataset"] in OPEN_DATASETS else "closed", axis=1
)
df["open_res"] = df.apply(
    lambda x: x["outcome"] if x["question_type"] == "open" else None, axis=1
)
df["closed_res"] = df.apply(
    lambda x: x["outcome"] if x["question_type"] == "closed" else None, axis=1
)

In [18]:
df.head()

,question_id,model,dataset,similarity,outcome,question_type,open_res,closed_res
0,0,internvl2,llava-bench,0.655310,60_80,open,60_80,None
1,1,internvl2,llava-bench,0.716396,60_80,open,60_80,None
2,2,internvl2,llava-bench,0.588038,60_80,open,60_80,None
3,3,internvl2,llava-bench,0.457952,60_80,open,60_80,None
4,4,internvl2,llava-bench,0.423745,40_60,open,40_60,None


In [ ]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,similarity,outcome,question_type,open_res,closed_res
197,3151,internvl2,seed,0.562908,correct,closed,None,correct
198,1656,internvl2,seed,0.448091,correct,closed,None,correct
199,154,internvl2,seed,0.711095,correct,closed,None,correct
200,3057,internvl2,seed,0.765221,correct,closed,None,correct
201,2173,internvl2,seed,0.429385,correct,closed,None,correct


Create equal ranges for cosine similarity

In [21]:
step = round(1 / 3, 3)
cosine_ranges = {
    idx: {"start": float(round(val, 3)), "end": float(round(val + step, 3))}
    for idx, val in enumerate(np.linspace(0, 1, 4))
}

# remove last range, meaningless here
cosine_ranges.pop(3)
cosine_ranges

{0: {'start': 0.0, 'end': 0.333},
 1: {'start': 0.333, 'end': 0.666},
 2: {'start': 0.667, 'end': 1.0}}

In [22]:
counts_per_model = {m: {ds: {} for ds in ds_list} for m in model_list}

for m, ds in zip(all_models, all_ds):
    for r in cosine_ranges:
        counts_per_model[m][ds][r] = len(
            df[
                (df["dataset"] == ds)
                & (df["model"] == m)
                & (df["similarity"] >= cosine_ranges[r]["start"])
                & (df["similarity"] <= cosine_ranges[r]["end"])
            ]
        )

In [23]:
counts_per_model

{'internvl2': {'llava-bench': {0: 2, 1: 44, 2: 12},
  'mmbench': {0: 8, 1: 81, 2: 50},
  'seed': {0: 17, 1: 93, 2: 40},
  'vqav2': {0: 6, 1: 89, 2: 54}},
 'llava-1.6': {'llava-bench': {0: 8, 1: 25, 2: 11},
  'mmbench': {0: 12, 1: 81, 2: 40},
  'seed': {0: 18, 1: 82, 2: 29},
  'vqav2': {0: 14, 1: 100, 2: 28}},
 'minigpt4': {'llava-bench': {0: 12, 1: 31, 2: 11},
  'mmbench': {0: 17, 1: 59, 2: 25},
  'seed': {0: 20, 1: 97, 2: 24},
  'vqav2': {0: 5, 1: 78, 2: 38}},
 'sharegpt4v': {'llava-bench': {0: 6, 1: 33, 2: 10},
  'mmbench': {0: 10, 1: 80, 2: 44},
  'seed': {0: 30, 1: 61, 2: 18},
  'vqav2': {0: 1, 1: 15, 2: 5}}}

In [24]:
counts_per_type = {idx: {} for idx in cosine_ranges}

for idx, r in cosine_ranges.items():
    for m, ds in zip(all_models, all_ds):
        if m not in counts_per_type[idx]:
            counts_per_type[idx][m] = {}
        if ds not in counts_per_type[idx][m]:
            counts_per_type[idx][m][ds] = 0.0

        counts_per_type[idx][m][ds] = len(
            df[
                (df["dataset"] == ds)
                & (df["model"] == m)
                & (df["similarity"] >= r["start"])
                & (df["similarity"] <= r["end"])
            ]
        )

In [25]:
counts_per_type

{0: {'internvl2': {'llava-bench': 2, 'mmbench': 8, 'seed': 17, 'vqav2': 6},
  'llava-1.6': {'llava-bench': 8, 'mmbench': 12, 'seed': 18, 'vqav2': 14},
  'minigpt4': {'llava-bench': 12, 'mmbench': 17, 'seed': 20, 'vqav2': 5},
  'sharegpt4v': {'llava-bench': 6, 'mmbench': 10, 'seed': 30, 'vqav2': 1}},
 1: {'internvl2': {'llava-bench': 44, 'mmbench': 81, 'seed': 93, 'vqav2': 89},
  'llava-1.6': {'llava-bench': 25, 'mmbench': 81, 'seed': 82, 'vqav2': 100},
  'minigpt4': {'llava-bench': 31, 'mmbench': 59, 'seed': 97, 'vqav2': 78},
  'sharegpt4v': {'llava-bench': 33, 'mmbench': 80, 'seed': 61, 'vqav2': 15}},
 2: {'internvl2': {'llava-bench': 12, 'mmbench': 50, 'seed': 40, 'vqav2': 54},
  'llava-1.6': {'llava-bench': 11, 'mmbench': 40, 'seed': 29, 'vqav2': 28},
  'minigpt4': {'llava-bench': 11, 'mmbench': 25, 'seed': 24, 'vqav2': 38},
  'sharegpt4v': {'llava-bench': 10, 'mmbench': 44, 'seed': 18, 'vqav2': 5}}}

In [28]:
counts_plot = {idx: {} for idx in cosine_ranges}

for idx, r in cosine_ranges.items():
    for m, ds in zip(all_models, all_ds):
        if ds not in counts_plot[idx]:
            counts_plot[idx][ds] = []

        counts_plot[idx][ds].append(
            len(
                df[
                    (df["dataset"] == ds)
                    & (df["model"] == m)
                    & (df["similarity"] >= r["start"])
                    & (df["similarity"] <= r["end"])
                ]
            )
        )

# Aligning the data for plotting
# Swap: 0 (Type 3 failures) <-> 2 (Type 1 failures)
# Good: 1 (Type 2 failures)

temp = counts_plot[0]
counts_plot[0] = counts_plot[2]
counts_plot[2] = temp

In [29]:
counts_plot

{0: {'llava-bench': [12, 11, 11, 10],
  'mmbench': [50, 40, 25, 44],
  'seed': [40, 29, 24, 18],
  'vqav2': [54, 28, 38, 5]},
 1: {'llava-bench': [44, 25, 31, 33],
  'mmbench': [81, 81, 59, 80],
  'seed': [93, 82, 97, 61],
  'vqav2': [89, 100, 78, 15]},
 2: {'llava-bench': [2, 8, 12, 6],
  'mmbench': [8, 12, 17, 10],
  'seed': [17, 18, 20, 30],
  'vqav2': [6, 14, 5, 1]}}

In [ ]:
out_path = Path(CHARTS_DIR, "behaviour_counts.pdf")
vis.plot_behaviour_counts(counts_plot, out_path)

/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:446: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
